# Imports

In [1]:
import pandas as pd
import numpy as np
from multiprocessing import Pool
import random
import re

#### 1. Tree Structure

In [2]:
class InnerNode:
    def __init__(self, name):
        self.name = name
        self.connectionOne = None
        self.connectionTwo = None
        self.connectionThree = None
        self.tempConnections = []
        self.hasInnerConnection = False
        
    def add_inner_node(self,node):
        self.tempConnections.append(node)
        self.hasInnerConnection = True
        
    def add_leaf(self,leaf):
        self.tempConnections.append(leaf)
        
    def is_not_full(self):
        return len(self.tempConnections) < 3
    
class LeafNode:
    def __init__(self, name):
        self.name = name
        self.innerConnection = None
        
    def add_node(self,node):
        self.innerConnection = node

#### 2. Generate Random Tree

In [3]:
def generate_random_tree(leaf_labels):
    
    """
        Generates a random tree from leaf_labels.

        Parameters
        ----------
        df : array_like
            A list with the labels of the objects to be put in a trenary tree structure.

        Returns
        -------
        tree_dict : dict
            A dictionary containing all the inner tree nodes and its connections to other inner nodes/ leafs.

        Notes
        --------
        Function returns error if leafs are less than 6.
    """
    
    if len(leaf_labels) < 6:
        raise Exception("There must be at least six objects to cluster.")
    
    inner_node_labels = np.array([])
    tree_dict = {}

    # Initialize inner nodes
    for n in range(0,len(leaf_labels)-2):
        inner_node_labels = np.append(inner_node_labels, ('n' + str(n)))

    # Tree dict
    for n in inner_node_labels:
        tree_dict[n] = np.array([])

    # print(f'leaf_lables: {leaf_labels}')
    # print(f'inner_node_labels: {inner_node_labels}')
    # print(f'tree_dict: {tree_dict}')

    # Create inner_node connections
    nodes_to_use = inner_node_labels.copy()
    for index,node in enumerate(inner_node_labels):

        if len(nodes_to_use) == 0:
            break

        if node in nodes_to_use:
            index = np.argwhere(nodes_to_use == node)
            nodes_to_use = np.delete(nodes_to_use, index)

        # Initialize root node
        if index == 0:

            # Roll for number of inner_node connections
            possible_rolls = np.arange(1,4)
            number_of_connections = np.random.choice(possible_rolls)

            # Create number of connections rolled
            for connection in range(number_of_connections):
                node_to_connect = nodes_to_use[0]
                nodes_to_use = np.delete(nodes_to_use, 0)
                tree_dict[node] = np.append(tree_dict[node], node_to_connect)
                tree_dict[node_to_connect] = np.append(tree_dict[node_to_connect], node)

        elif index != 0 and len(tree_dict[node]) < 3:

            # Roll for number of inner_node connection
            possible_rolls = np.arange(1,4-len(tree_dict[node]))
            number_of_connections = np.random.choice(possible_rolls)

            if number_of_connections > len(nodes_to_use):
                number_of_connections = len(nodes_to_use)

            # Create number of connections rolled
            for connection in range(number_of_connections):
                node_to_connect = nodes_to_use[0]
                nodes_to_use = np.delete(nodes_to_use, 0)
                tree_dict[node] = np.append(tree_dict[node], node_to_connect)
                tree_dict[node_to_connect] = np.append(tree_dict[node_to_connect], node)

    # Add leafs to inner_nodes
    for node in tree_dict.keys():

        number_of_leafs = range(3 - len(tree_dict[node]))

        for n in number_of_leafs:
            leaf = np.random.choice(leaf_labels,1)
            leaf_index = np.where(leaf_labels == leaf)
            leaf_labels = np.delete(leaf_labels,leaf_index)
            tree_dict[node] = np.append(tree_dict[node], leaf)
            
        
            
    return tree_dict

In [4]:
tree_dict = generate_random_tree(['a','b','c','d','e','f','g','h'])
tree_dict

{'n0': array(['n1', 'n2', 'c'], dtype='<U32'),
 'n1': array(['n0', 'n3', 'g'], dtype='<U32'),
 'n2': array(['n0', 'n4', 'd'], dtype='<U32'),
 'n3': array(['n1', 'n5', 'f'], dtype='<U32'),
 'n4': array(['n2', 'b', 'h'], dtype='<U32'),
 'n5': array(['n3', 'a', 'e'], dtype='<U32')}

#### 3. Leaf Swap

In [5]:
def select_two_random_nodes_with_leafs(tree_dict):
    
    """
        Selects two random inner nodes.

        Parameters
        ----------
        tree_dict : dict
            A dictionary containing all the inner tree nodes and its connections to other inner nodes/ leafs.

        Returns
        -------
        random_nodes : array_like
            A list with the names of two random inner nodes.
        
    """
    # Get list of nodes
    nodes = np.array(list(tree_dict.keys()))
    
    # Regex for: starts with "n" and the next 1-5 characters are digits from 0-9
    pattern = "^[n][0-9]{1,5}$"
    
    # Initialize random nodes
    first_random_node = None
    second_random_node = None
    
    # Initialize leaf arrays
    leaf_indexes_1 = np.array([])
    leaf_indexes_2 = np.array([])
    
    # Search for first node with leafs
    while len(leaf_indexes_1) == 0:
        first_random_node = np.random.choice(nodes,1)
        for i,node in enumerate(tree_dict[first_random_node[0]]):
            if not re.match(pattern, node):
                leaf_indexes_1 = np.append(leaf_indexes_1,i)
                
    # Remove first random node from the options
    index = np.argwhere(nodes == first_random_node)
    nodes = np.delete(nodes, index)
    
    # Search for second node with leafs
    while len(leaf_indexes_2) == 0:
        second_random_node = np.random.choice(nodes,1)
        for i,node in enumerate(tree_dict[second_random_node[0]]):
            if not re.match(pattern, node):
                leaf_indexes_2 = np.append(leaf_indexes_2,i)
            
    
    return {first_random_node[0]: leaf_indexes_1, second_random_node[0]: leaf_indexes_2}

In [6]:
select_two_random_nodes_with_leafs(tree_dict)

{'n2': array([2.]), 'n4': array([1., 2.])}

In [7]:
def leaf_swap(tree_dict):
    
    """
        Swap two leaf labels of given tree.

        Parameters
        ----------
        tree_dict : dict
            A dictionary with all the inner nodes and their respective connections.

        Returns
        -------
        tree_dict : dict
            A dictionary containing all the inner tree nodes and its connections to other inner nodes/ leafs, with two random leafs swaped.

        Notes
        --------
        Random mother nodes can be connected.
    """
    
    # Select two random unconnected inner nodes
    random_nodes = select_two_random_nodes_with_leafs(tree_dict)
    
    # Select two random leafs to switch
    nodes = list(random_nodes.keys())
    print(f'nodes:{nodes}')
    random_leaf_1 = np.random.choice(random_nodes[nodes[0]])
    random_leaf_2 = np.random.choice(random_nodes[nodes[1]])
    print(tree_dict[nodes[0]][int(random_leaf_1)], tree_dict[nodes[1]][int(random_leaf_2)])
    
    # Save leaf values
    leaf_1_val = tree_dict[nodes[0]][int(random_leaf_1)]
    leaf_2_val = tree_dict[nodes[1]][int(random_leaf_2)]
    
    # Switch leaf values
    tree_dict[nodes[0]][int(random_leaf_1)] = leaf_2_val
    tree_dict[nodes[1]][int(random_leaf_2)] = leaf_1_val
    
    return tree_dict

leaf_swap(tree_dict)

nodes:['n1', 'n2']
g d


{'n0': array(['n1', 'n2', 'c'], dtype='<U32'),
 'n1': array(['n0', 'n3', 'd'], dtype='<U32'),
 'n2': array(['n0', 'n4', 'g'], dtype='<U32'),
 'n3': array(['n1', 'n5', 'f'], dtype='<U32'),
 'n4': array(['n2', 'b', 'h'], dtype='<U32'),
 'n5': array(['n3', 'a', 'e'], dtype='<U32')}

#### 4. Swap inner nodes

In [8]:
def select_two_unconected_nodes(tree_dict):
    
    """
        Selects two random (unconnected) inner nodes.

        Parameters
        ----------
        tree_dict : dict
            A dictionary containing all the inner tree nodes and its connections to other inner nodes/ leafs.

        Returns
        -------
        random_nodes : array_like
            A list with the names of two radom unconnected inner nodes.

    """
    
    # Select a random node
    nodes = list(tree_dict.keys())
    first_random_node = np.random.choice(nodes,1)
    
    # Remove random node and its connections from pool of nodes before choosing another
    nodes_to_remove = np.append([first_random_node[0]],tree_dict[first_random_node[0]])
    new_nodes = np.array([])
    
    # Create new node list containing only unconnected nodes
    for node in nodes:
        if node not in nodes_to_remove:
            new_nodes = np.append(new_nodes,node)
    
    # Select seccond random node
    second_random_node = np.random.choice(new_nodes,1)
    
    return [first_random_node, second_random_node]

In [22]:
def swap_inner_nodes(tree_dict):
    

    
    # Initialize nodes
    nodes = list(tree_dict.keys())

    # Find two nodes with swapable inner connections
    find_nodes = True
    random_nodes = None
    while find_nodes:

        # Choose two random nodes
        random_nodes = np.random.choice(nodes,2,replace=False)
        # print(f'random nodes:{random_nodes}')

        # Check if nodes are connected and remove connection if there is any
        nodes_zero = np.array(tree_dict[random_nodes[0]])
        nodes_one = np.array(tree_dict[random_nodes[1]])

        if random_nodes[1] in tree_dict[random_nodes[0]]:
            # print('yep')
            index = np.argwhere(nodes_zero == random_nodes[1])
            nodes_zero = np.delete(nodes_zero, index)

            index = np.argwhere(nodes_one == random_nodes[0])
            nodes_one = np.delete(nodes_one, index)

        # Regex for: starts with "n" and the next 1-5 characters are digits from 0-9
        pattern = "^[n][0-9]{1,5}$"

        # Check if there are any inner connection
        inner_connections_zero = []
        inner_connections_one = []

        # Node one
        for node in nodes_zero:
            if re.match(pattern, node):
                inner_connections_zero.append(node)

        # Node two
        for node in nodes_one:
            if re.match(pattern, node):
                inner_connections_one.append(node)

        # Remove common connection

        # Node zero
        new_nodes_zero = []
        for node in inner_connections_zero:
             if node not in inner_connections_one:
                new_nodes_zero.append(node)

        # Node one
        new_nodes_one = []
        for node in inner_connections_one:
             if node not in inner_connections_zero:
                new_nodes_one.append(node)

        if len(new_nodes_zero) > 0 and len(new_nodes_one) > 0:
            find_nodes = False

    # Select nodes to swap
    choices = [np.random.choice(new_nodes_zero,1), np.random.choice(new_nodes_one,1)]

    # print(choices[0], choices[1])

    # Swap inner connections
    index = np.argwhere(tree_dict[random_nodes[0]] == choices[0][0])
    tree_dict[random_nodes[0]][index] = choices[1][0]

    index = np.argwhere(tree_dict[choices[0][0]] == random_nodes[0])
    tree_dict[choices[0][0]][index] = random_nodes[1]

    index = np.argwhere(tree_dict[random_nodes[1]] == choices[1][0])
    tree_dict[random_nodes[1]][index] = choices[0][0]

    index = np.argwhere(tree_dict[choices[1][0]] == random_nodes[1])
    tree_dict[choices[1][0]][index] = random_nodes[0]

    return tree_dict


In [23]:
swap_inner_nodes(tree_dict)

{'n0': array(['n1', 'n2', 'c'], dtype='<U32'),
 'n1': array(['n0', 'n3', 'd'], dtype='<U32'),
 'n2': array(['n0', 'n5', 'g'], dtype='<U32'),
 'n3': array(['n4', 'n1', 'f'], dtype='<U32'),
 'n4': array(['n3', 'b', 'h'], dtype='<U32'),
 'n5': array(['n2', 'a', 'e'], dtype='<U32')}

In [64]:
def Checker(reader):
    return np.unique(reader).size != reader.size

In [75]:
error = False
for i in range(1000):
    swap_inner_nodes(tree_dict)
    for k,v in tree_dict.items():
        if Checker(v) == True:
            error = True

print(error)

False


#### 5. Swap Node with leaf

In [171]:
def swap_node_with_leaf(tree_dict):
    
    """
        Swap node with leaf.

        Parameters
        ----------
        tree_dict : dict
            A dictionary with all the inner nodes and their respective connections.

        Returns
        -------
        tree_dict : dict
            A dictionary containing all the inner tree nodes and its connections to other inner nodes/ leafs, with a leaf and a node swapped.

        Notes
        --------
        Leaf node must have at least 1 leaf, since one is needed for the swapping process. Inner node must have at least 2 inner connections since
        the node recieving a leaf always needs to keep one inner connection to remain connected to the tree.
    """
    
    # Initialize nodes
    nodes = list(tree_dict.keys())
    
    # Regex for: starts with "n" and the next 1-5 characters are digits from 0-9
    pattern = "^[n][0-9]{1,5}$"
    
    # Select node with at least one leaf
    leaf_node = None
    leafs = np.array([])
    
    # Search for first node with leafs
    while len(leafs) == 0:
        leaf_node = np.random.choice(nodes,1)
        for node in tree_dict[leaf_node[0]]:
            if not re.match(pattern, node):
                leafs = np.append(leafs, node)
    
    # Remove first random node from the options
    index = np.argwhere(nodes == leaf_node)
    nodes = np.delete(nodes, index)
    
    # Search for node with at least 2 inner connections
    inner_node = None
    inner_connections = np.array([])
    while len(inner_connections) < 2:
        inner_node = np.random.choice(nodes,1)
        inner_connections = np.array([])
        for node in tree_dict[inner_node[0]]:
            if re.match(pattern, node):
                inner_connections = np.append(inner_connections, node)
                
    # Remove connection between nodes, if there is any
    index = np.argwhere(inner_connections  == leaf_node)
    if len(index) > 0:
        inner_connections = np.delete(inner_connections, index)
    
    print(f'leaf_node: {leaf_node} \t leafs: {leafs}')
    print(f'inner_node: {inner_node} \t inner_connections: {inner_connections}')
    
    random_leaf = np.random.choice(leafs,1)
    random_inner_node = np.random.choice(inner_connections,1)
    
    print(f'random_leaf: {random_leaf}')
    print(f'random_inner_node : {random_inner_node}')
    
    
    index = np.argwhere(tree_dict[leaf_node[0]] == random_leaf)
    tree_dict[leaf_node[0]][index] = random_inner_node
    
    index = np.argwhere(tree_dict[random_inner_node[0]] == inner_node)
    tree_dict[random_inner_node[0]][index] = leaf_node
    
    index = np.argwhere(tree_dict[inner_node[0]] == random_inner_node)
    tree_dict[inner_node[0]][index] = random_leaf
    
    return tree_dict


In [176]:
tree_dict = generate_random_tree(['a','b','c','d','e','f','g','h'])
tree_dict

{'n0': array(['n1', 'n2', 'g'], dtype='<U32'),
 'n1': array(['n0', 'n3', 'n4'], dtype='<U32'),
 'n2': array(['n0', 'n5', 'e'], dtype='<U32'),
 'n3': array(['n1', 'd', 'h'], dtype='<U32'),
 'n4': array(['n1', 'a', 'f'], dtype='<U32'),
 'n5': array(['n2', 'b', 'c'], dtype='<U32')}

In [179]:
swap_node_with_leaf(tree_dict)

leaf_node: n4 	 leafs: ['a' 'f']
inner_node: ['n3'] 	 inner_connections: ['n1' 'n0']
random_leaf: ['a']
random_inner_node : ['n0']


{'n0': array(['c', 'n4', 'g'], dtype='<U32'),
 'n1': array(['n5', 'n3', 'n4'], dtype='<U32'),
 'n2': array(['h', 'n5', 'e'], dtype='<U32'),
 'n3': array(['n1', 'd', 'a'], dtype='<U32'),
 'n4': array(['n1', 'n0', 'f'], dtype='<U32'),
 'n5': array(['n2', 'b', 'n1'], dtype='<U32')}